In [1]:
import pandas as pd
import json

In [2]:
contrib = pd.read_csv('package_data_final_with_scores_contrib_fixed.csv')
maintained = pd.read_csv('package_data_final_with_scores_maintained.csv')
dependUpTool = pd.read_csv('package_data_final_with_scores_dut_fixed.csv')

fuzzing = pd.read_csv('fuzzing_subset.csv')

## Maintained

In [3]:
def safe_parse(x):
    try:
        return json.loads(x)
    except (json.JSONDecodeError, TypeError):
        return None

maintained['scorecard_parsed'] = maintained['scorecard'].apply(safe_parse)

# Check how many failed
print(maintained['scorecard_parsed'].isna().sum(), "rows with invalid scorecard")

1735 rows with invalid scorecard


In [4]:
# Print the repos that failed
print(maintained[maintained['scorecard_parsed'].isna()][['githubrepo', 'scorecard']])

# Print the number of unique repos that failed
print(maintained[maintained['scorecard_parsed'].isna()]['githubrepo'].nunique(), "unique repos with invalid scorecard")

# Print in a way that I can paste into an excel column
print("\n".join(maintained[maintained['scorecard_parsed'].isna()]['githubrepo'].unique()))

                                            githubrepo              scorecard
30007   https://github.com/faros-ai/airbyte-connectors  ERROR 404 UNREACHABLE
30008   https://github.com/faros-ai/airbyte-connectors  ERROR 404 UNREACHABLE
30009   https://github.com/faros-ai/airbyte-connectors  ERROR 404 UNREACHABLE
30010   https://github.com/faros-ai/airbyte-connectors  ERROR 404 UNREACHABLE
30011   https://github.com/faros-ai/airbyte-connectors  ERROR 404 UNREACHABLE
...                                                ...                    ...
365884       https://github.com/lucidrains/ema-pytorch  ERROR 404 UNREACHABLE
365885       https://github.com/lucidrains/ema-pytorch  ERROR 404 UNREACHABLE
365886       https://github.com/lucidrains/ema-pytorch  ERROR 404 UNREACHABLE
365887       https://github.com/lucidrains/ema-pytorch  ERROR 404 UNREACHABLE
365888       https://github.com/lucidrains/ema-pytorch  ERROR 404 UNREACHABLE

[1735 rows x 2 columns]
27 unique repos with invalid scorecard


In [5]:
# Extract fields where parsing succeeded
mask = maintained['scorecard_parsed'].notna()
maintained.loc[mask, 'Maintained_score'] = maintained.loc[mask, 'scorecard_parsed'].apply(lambda x: x['score'])
#maintained.loc[mask, 'checks_score'] = maintained.loc[mask, 'scorecard_parsed'].apply(lambda x: x['checks'][0]['score'])
maintained.loc[mask, 'Maintained_reason'] = maintained.loc[mask, 'scorecard_parsed'].apply(lambda x: x['checks'][0]['reason'])
maintained.loc[mask, 'Maintained_details'] = maintained.loc[mask, 'scorecard_parsed'].apply(lambda x: x['checks'][0]['details'])

## Dependency Update Tool 

In [6]:
dependUpTool['scorecard_parsed'] = dependUpTool['scorecard'].apply(safe_parse)

# Check how many failed
print(dependUpTool['scorecard_parsed'].isna().sum(), "rows with invalid scorecard")

1384 rows with invalid scorecard


In [7]:
# Print the repos that failed
print(dependUpTool[dependUpTool['scorecard_parsed'].isna()][['github_repo', 'scorecard']])

# Print the number of unique repos that failed
print(dependUpTool[dependUpTool['scorecard_parsed'].isna()]['github_repo'].nunique(), "unique repos with invalid scorecard")

# Print in a way that I can paste into an excel column
print("\n".join(dependUpTool[dependUpTool['scorecard_parsed'].isna()]['github_repo'].unique()))

                                     github_repo scorecard
19418   https://github.com/uladkasach/declapract       NaN
19419   https://github.com/uladkasach/declapract       NaN
19420   https://github.com/uladkasach/declapract       NaN
19421   https://github.com/uladkasach/declapract       NaN
19422   https://github.com/uladkasach/declapract       NaN
...                                          ...       ...
365244  https://github.com/softvisio-node/argon2       NaN
365245  https://github.com/softvisio-node/argon2       NaN
365246  https://github.com/softvisio-node/argon2       NaN
365247  https://github.com/softvisio-node/argon2       NaN
365248  https://github.com/softvisio-node/argon2       NaN

[1384 rows x 2 columns]
27 unique repos with invalid scorecard
https://github.com/uladkasach/declapract
https://github.com/cloudgraphdev/cloudgraph-provider-aws
https://github.com/faros-ai/airbyte-connectors
https://github.com/jeremydaly/dynamodb-toolbox
https://github.com/nifcloud/nifcloud

In [8]:
# Extract fields where parsing succeeded
mask = dependUpTool['scorecard_parsed'].notna()
dependUpTool.loc[mask, 'DependUpTool_score'] = dependUpTool.loc[mask, 'scorecard_parsed'].apply(lambda x: x['score'])
#dependUpTool.loc[mask, 'checks_score'] = dependUpTool.loc[mask, 'scorecard_parsed'].apply(lambda x: x['checks'][0]['score'])
dependUpTool.loc[mask, 'DependUpTool_reason'] = dependUpTool.loc[mask, 'scorecard_parsed'].apply(lambda x: x['checks'][0]['reason'])
dependUpTool.loc[mask, 'DependUpTool_details'] = dependUpTool.loc[mask, 'scorecard_parsed'].apply(lambda x: x['checks'][0]['details'])

## Contributors

In [9]:
import json
import pandas as pd


def safe_parse_scorecard(x):
    if pd.isna(x) or not isinstance(x, str):
        return None

    try:
        # 1. Split on the first two commas to isolate the JSON block
        parts = x.split(",", maxsplit=2)
        if len(parts) < 3:
            return None

        json_string = parts[2].strip()

        # 2. Clean up the CSV double-quoting mess
        # Remove the outer quotes if they exist
        if json_string.startswith('"') and json_string.endswith('"'):
            json_string = json_string[1:-1]

        # Convert double-double quotes ("") to single-double quotes (")
        json_string = json_string.replace('""', '"')

        # 3. Parse the clean JSON string
        return json.loads(json_string)

    except (json.JSONDecodeError, TypeError, IndexError) as e:
        # Uncomment the line below if you want to debug exactly why it failed
        # print(f"Failed parsing due to: {e} | Raw JSON text: {json_string[:50]}")
        return None


# Apply the fix
contrib["scorecard_parsed"] = contrib["scorecard"].apply(
    safe_parse_scorecard
)

# Check how many failed now
failed_count = contrib["scorecard_parsed"].isna().sum()
print(f"{failed_count} rows out of {len(contrib)} failing.")

2092 rows out of 367102 failing.


In [10]:
# Print the repos that failed
print(contrib[contrib['scorecard_parsed'].isna()][['githubrepo', 'scorecard']])

# Print the number of unique repos that failed
print(contrib[contrib['scorecard_parsed'].isna()]['githubrepo'].nunique(), "unique repos with invalid scorecard")

# Print in a way that I can paste into an excel column
print("\n".join(contrib[contrib['scorecard_parsed'].isna()]['githubrepo'].unique()))

                                              githubrepo  \
464     https://github.com/authsignal/authsignal-browser   
22391              https://github.com/jupyter/jupyterlab   
22392              https://github.com/jupyter/jupyterlab   
22393              https://github.com/jupyter/jupyterlab   
22394              https://github.com/jupyter/jupyterlab   
...                                                  ...   
365884         https://github.com/lucidrains/ema-pytorch   
365885         https://github.com/lucidrains/ema-pytorch   
365886         https://github.com/lucidrains/ema-pytorch   
365887         https://github.com/lucidrains/ema-pytorch   
365888         https://github.com/lucidrains/ema-pytorch   

                                                scorecard  
464         @authsignal/browser,NPM,ERROR 404 UNREACHABLE  
22391    'Connection to api.github.com timed out. (con...  
22392    'Connection to api.github.com timed out. (con...  
22393    'Connection to api.github.com 

In [11]:
# Extract fields where parsing succeeded
mask = contrib['scorecard_parsed'].notna()
contrib.loc[mask, 'Contrib_score'] = contrib.loc[mask, 'scorecard_parsed'].apply(lambda x: x['score'])
#contrib.loc[mask, 'checks_score'] = contrib.loc[mask, 'scorecard_parsed'].apply(lambda x: x['checks'][0]['score'])
contrib.loc[mask, 'Contrib_reason'] = contrib.loc[mask, 'scorecard_parsed'].apply(lambda x: x['checks'][0]['reason'])
contrib.loc[mask, 'Contrib_details'] = contrib.loc[mask, 'scorecard_parsed'].apply(lambda x: x['checks'][0]['details'])

In [12]:
## delete the scorecard_parsed column since we don't need it anymore
contrib = contrib.drop(columns=['scorecard_parsed'])
maintained = maintained.drop(columns=['scorecard_parsed'])
dependUpTool = dependUpTool.drop(columns=['scorecard_parsed'])

## delete the scorecard column since we don't need it anymore
contrib = contrib.drop(columns=['scorecard'])
maintained = maintained.drop(columns=['scorecard'])
dependUpTool = dependUpTool.drop(columns=['scorecard'])

## Create csv files of the parsed data

In [13]:
contrib.to_csv('contrib_expanded.csv', index=False)
maintained.to_csv('maintained_expanded.csv', index=False)
dependUpTool.to_csv('dependUpTool_expanded.csv', index=False)

In [14]:
# need to merge the datasets 
maintained = maintained.drop(columns=["published_at", "status", "release_tag_name", "project_name", "System", "package_name"])

merged_data = pd.merge(contrib, 
                       maintained,
                       on=['githubrepo', 'tag_name', 'tag_commit_sha'])

In [15]:
dependUpTool = dependUpTool.drop(columns=["published_at", "project_name", "System", "package_name"])

merged_data_2 = pd.merge(merged_data, 
                       dependUpTool,
                       left_on=['githubrepo', 'tag_name', 'tag_commit_sha'],
                       right_on=['github_repo', 'tag_name', 'tag_commit_sha'])

In [16]:
## now we need to check for duplicated rows and remove them
print(f"Number of rows before merge: {len(merged_data_2)}")  

merged_data_2 = merged_data_2.drop_duplicates(subset=['githubrepo', 'tag_name', 'tag_commit_sha'], keep='first')

## print the number of duplicates before and after removing duplicates
print(f"Number of rows after merge: {len(merged_data_2)}")  

merged_data_2 = merged_data_2.drop(columns=['github_repo'])

Number of rows before merge: 376220
Number of rows after merge: 365580


In [17]:
## need to add fuzzing data to the merged dataset
fuzzing = fuzzing.drop(columns=["status", "release_tag_name", "System"])

final_merged_data = pd.merge(merged_data_2,
                             fuzzing,
                             left_on=['githubrepo', 'tag_name', 'tag_commit_sha'],
                             right_on=['github_repo', 'tag_name', 'tag_commit_sha'],
                             how = 'outer')

In [18]:
final_merged_data = final_merged_data.drop(columns=['github_repo'])

In [19]:
final_merged_data.to_csv('final_nonlocal_data.csv', index=False)